<a href="https://colab.research.google.com/github/manoelfs/Job_hunting/blob/main/Job_Hunting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Robo para coletar vagas
Esse robô tem como objetivo varrer os principais sites de vagas e incluir no excel as vagas que correnpondes aos parametros informados


# ⚙️ Configurações Globais
Nesta seção, definimos as variáveis e parâmetros que serão utilizados por todo o pipeline de busca e integração.

In [51]:
# ==========================================
# CONFIGURAÇÕES GLOBAIS DE BUSCA DE VAGAS
# ==========================================

# Altere aqui o cargo ou termo que deseja pesquisar nas plataformas
PALAVRA_CHAVE = "SRE remoto"

# Defina a localidade geográfica para a pesquisa (garante foco no Brasil)
LOCALIDADE = "Brasil"

# Definido como None para trazer o máximo de vagas possíveis encontradas na busca
LIMITE_VAGAS = None

In [ ]:
# Instalação do Playwright e dependências necessárias
!pip install -q playwright nest_asyncio
!playwright install chromium
!playwright install-deps

Installing dependencies...
Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Hit:5 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Hit:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Get:9 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,409 kB]
Get:10 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,166 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,729 kB]
Fetched 5,560 kB in 2s (3,105 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu

# 🔗 Coleta de Vagas - LinkedIn
A partir daqui, todas as etapas, instalações e funções são voltadas especificamente para a extração de vagas da plataforma LinkedIn.

### 📋 Etapas do Processo de Coleta

1. **Definição dos Parâmetros de Busca** (Herdados das Configurações Globais)
2. **Inicialização do Navegador** (Utilizando o Playwright configurado anteriormente)
3. **Execução do Crawler**:
    * Acesso à página de busca pública do LinkedIn
    * Simulação de comportamento humano (scroll)
    * Extração de títulos, empresas, links e resumos
4. **Armazenamento de Dados** (Salvamento estruturado no Google Drive)

In [ ]:
import sys
import nest_asyncio
import asyncio

# Garante que os caminhos das bibliotecas recém-instaladas sejam atualizados no sys.path
import site
site.main()

from playwright.async_api import async_playwright

# Aplica o patch para permitir loops assíncronos aninhados no Colab
nest_asyncio.apply()

async def testar_navegador():
    async with async_playwright() as p:
        # Lança o navegador Chromium em modo headless
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        # Acessa uma URL simples de teste
        url_teste = "https://httpbin.org/ip"
        print(f"Acessando {url_teste} via Playwright...")
        await page.goto(url_teste)

        conteudo = await page.content()
        print("Conteúdo retornado pelo navegador com sucesso:")
        print(conteudo)

        await browser.close()

# Executa o teste de inicialização do browser
try:
    asyncio.run(testar_navegador())
except Exception as e:
    print(f"Ocorreu um erro ao rodar o teste: {e}")

Acessando https://httpbin.org/ip via Playwright...
Conteúdo retornado pelo navegador com sucesso:
<html><head><meta name="color-scheme" content="light dark"><meta charset="utf-8"></head><body><pre>{
  "origin": "35.231.106.48"
}
</pre><div class="json-formatter-container"></div></body></html>


In [ ]:
import os

# Defina o caminho da sua pasta no Google Drive
pasta_destino = "/content/drive/MyDrive/Colab Notebooks/Vagas"
nome_arquivo = "vagas_mapeadas_aderencia.xlsx"

# Use a função os.path.join para combinar o caminho com o nome do arquivo
caminho_completo = os.path.join(pasta_destino, nome_arquivo)

print("Caminho da pasta:", pasta_destino)
print("Nome do arquivo:", nome_arquivo)
print("-" * 50)
print("Resultado combinado:", caminho_completo)

Caminho da pasta: /content/drive/MyDrive/Colab Notebooks/Vagas
Nome do arquivo: vagas_mapeadas_aderencia.xlsx
--------------------------------------------------
Resultado combinado: /content/drive/MyDrive/Colab Notebooks/Vagas/vagas_mapeadas_aderencia.xlsx


In [ ]:
import os

pasta_teste = "/content/drive/MyDrive/Colab Notebooks/Vagas"

# Verifica se o caminho da pasta NÃO existe
if not os.path.exists(pasta_teste):
    print(f"A pasta '{pasta_teste}' não existe. Criando agora...")
    # Cria a pasta de forma segura
    os.makedirs(pasta_teste, exist_ok=True)
    print("Pasta criada com sucesso!")
else:
    print(f"A pasta '{pasta_teste}' já existe! Pronta para uso.")

A pasta '/content/drive/MyDrive/Colab Notebooks/Vagas' já existe! Pronta para uso.


In [54]:
import re
import json
import asyncio
import random
import os
import pandas as pd
import requests
from bs4 import BeautifulSoup
from google.colab import userdata
from playwright.async_api import async_playwright
import nest_asyncio

# 1. Configurações Globais
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    GOOGLE_API_KEY = None

# 2. Função de Extração de Vagas com Playwright (Com filtro estrito de localidade Brasil e filtro Remoto)
async def buscar_vagas_linkedin(palavra_chave="SRE remoto", localidade="Brasil", limite=None):
    vagas_listadas = []
    # f_WT=2 filtra especificamente por vagas remotas no LinkedIn
    url_busca = f"https://www.linkedin.com/jobs/search?keywords={requests.utils.quote(palavra_chave)}&location={requests.utils.quote(localidade)}&f_WT=2"

    async with async_playwright() as p:
        print(f"Iniciando navegador Playwright para buscar por: '{palavra_chave}' em '{localidade}' (Remoto)...")
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
            viewport={"width": 1920, "height": 1080},
            locale="pt-BR"
        )
        page = await context.new_page()

        try:
            await page.goto(url_busca, wait_until="domcontentloaded", timeout=45000)
            await page.wait_for_timeout(3000)

            # Simulação de scroll para carregar mais vagas
            print("⏬ Rolando a página para carregar as vagas disponíveis no Brasil...")
            for _ in range(5):
                await page.mouse.wheel(0, 1500)
                await page.wait_for_timeout(random.uniform(2000, 4000))

            html = await page.content()
            soup = BeautifulSoup(html, 'html.parser')

            cards = soup.select(".jobs-search__results-list li") or soup.find_all('div', class_=re.compile(r'base-card|job-search-card'))
            print(f"📊 Encontrados {len(cards)} elementos de vagas na busca do LinkedIn.")

            # Se o limite for None, trazemos todas as vagas identificadas
            limite_slice = len(cards) if limite is None else limite

            for card in cards[:limite_slice]:
                try:
                    title_el = card.select_one(".base-search-card__title, .job-search-card__title, h3")
                    company_el = card.select_one(".base-search-card__subtitle, .job-search-card__subtitle, h4")
                    link_el = card.select_one("a.base-card__full-link, a.job-search-card__link, a")
                    location_el = card.select_one(".job-search-card__location, .base-search-card__metadata, span")

                    titulo = title_el.text.strip() if title_el else "Não informado"
                    empresa = company_el.text.strip() if company_el else "Não informado"
                    local_vaga = location_el.text.strip() if location_el else ""

                    # Filtro de segurança adicional: ignorar vagas de fora do Brasil se detectadas textualmente
                    paises_estrangeiros = ["United States", "US", "Canada", "CA", "Europe", "Portugal", "Germany", "UK", "Argentina"]
                    if any(pais in local_vaga for pais in paises_estrangeiros) and "Brazil" not in local_vaga and "Brasil" not in local_vaga:
                        continue

                    link = ""
                    if link_el:
                        link = link_el.get('href', '')
                        link = link.split('?')[0] if '?' in link else link

                    descricao_resumo = f"Vaga de {titulo} na empresa {empresa} ({local_vaga}). Link direto para se candidatar: {link}"

                    if titulo != "Não informado":
                        vagas_listadas.append({
                            "titulo": titulo,
                            "empresa": empresa,
                            "link": link,
                            "descricao": descricao_resumo
                        })
                except Exception:
                    continue

        except Exception as e:
            print(f"Erro durante a navegação/scraping: {e}")
        finally:
            await browser.close()

    return vagas_listadas

# 3. Execução do Pipeline LinkedIn (Retorna DataFrame em memória sem salvar arquivo individual)
async def pipeline_principal_linkedin():
    vagas = await buscar_vagas_linkedin(palavra_chave=PALAVRA_CHAVE, localidade=LOCALIDADE, limite=LIMITE_VAGAS)

    if not vagas:
        print("\n⚠️ Nenhuma vaga capturada pelo crawler no momento. Usando vagas fictícias para manter a estrutura...")
        vagas = [
            {
                "titulo": "Engenheiro SRE Sênior (Remoto)",
                "empresa": "Cognitive Tech",
                "link": "https://exemplo.com/vaga-sre-senior",
                "descricao": "Atuar na sustentação, automação de infraestrutura híbrida e garantia de alta confiabilidade de nossos serviços de forma 100% remota."
            },
            {
                "titulo": "Especialista SRE / DevOps (Remoto)",
                "empresa": "Cloud Services Brazil",
                "link": "https://exemplo.com/vaga-sre-esp",
                "descricao": "Garantir a escalabilidade dos microsserviços e governança de observabilidade e telemetria atuando de casa."
            }
        ]

    resultados = []
    for v in vagas:
        resultados.append({
            "Título da Vaga": v['titulo'],
            "Empresa": v['empresa'],
            "Link": v['link'],
            "Descrição Resumida": v['descricao'],
            "Aderência (%)": "",
            "Justificativa": ""
        })

    df_final = pd.DataFrame(resultados)
    print(f"🟢 [LinkedIn] Coleta finalizada em memória: {len(df_final)} registros.")
    return df_final

nest_asyncio.apply()

## 🔗 Coleta de Vagas - Adzuna
A partir daqui, as etapas e funções são voltadas especificamente para a extração de vagas utilizando a API oficial do Adzuna.

In [55]:
import requests
import os
import pandas as pd

# Credenciais fornecidas para a API do Adzuna
ADZUNA_APP_ID = "6f7ed736"
ADZUNA_APP_KEY = "d00171133ba56442045322add932838e"

def buscar_vagas_adzuna(cargo, local, app_id, app_key, limite=None):
    """
    Consulta vagas de emprego de múltiplos portais através da API oficial do Adzuna.
    Retorna uma lista de dicionários estruturada.
    """
    print("🌐 [Adzuna API] Iniciando consulta de mercado...")
    vagas_coletadas = []

    # URL base corrigida da API do Adzuna para busca de vagas
    url_api = "https://api.adzuna.com/v1/api/jobs/br/search/1"

    # Parâmetros obrigatórios de autenticação e busca
    params = {
        "app_id": app_id,
        "app_key": app_key,
        "what": cargo,
        "where": local,
        "content-type": "application/json"
    }

    # Se houver um limite definido, adiciona aos parâmetros
    if limite is not None:
        params["results_per_page"] = limite
    else:
        params["results_per_page"] = 50  # Limite máximo padrão do Adzuna por requisição pública

    try:
        response = requests.get(url_api, params=params)

        if response.status_code == 200:
            dados = response.json()
            resultados = dados.get("results", [])

            for job in resultados:
                titulo = job.get("title", "N/A")
                titulo = titulo.replace("<strong>", "").replace("</strong>", "")

                vagas_coletadas.append({
                    "titulo": titulo,
                    "empresa": job.get("company", {}).get("display_name", "N/A"),
                    "link": job.get("redirect_url", "#"),
                    "descricao": job.get("description", "N/A")
                })
            print(f"✅ [Adzuna API] Varredura finalizada. {len(vagas_coletadas)} vagas encontradas bruto.")

        elif response.status_code == 403:
            print("❌ [Adzuna API] Erro 403: Falha de autenticação. Verifique se seu app_id e app_key estão corretos.")
        else:
            print(f"⚠️ [Adzuna API] Erro inesperado: Código de status {response.status_code}")

    except Exception as e:
        print(f"❌ [Adzuna API] Falha crítica de conexão: {e}")

    return vagas_coletadas

def pipeline_adzuna():
    # Executa a busca em memória sem salvar arquivo individual
    vagas = buscar_vagas_adzuna(
        cargo=PALAVRA_CHAVE,
        local=LOCALIDADE,
        app_id=ADZUNA_APP_ID,
        app_key=ADZUNA_APP_KEY,
        limite=LIMITE_VAGAS
    )

    if not vagas:
        print("\n⚠️ Nenhuma vaga capturada pela API do Adzuna no momento. Usando vagas fictícias estruturadas...")
        vagas = [
            {
                "titulo": f"Coordenador de Infraestrutura e TI ({PALAVRA_CHAVE})",
                "empresa": "Tech Solutions Brazil",
                "link": "https://adzuna.com/exemplo-vaga-coordenador",
                "descricao": "Responsável por coordenar equipes de infraestrutura, suporte técnico e projetos de nuvem."
            },
            {
                "titulo": f"Líder Técnico de Engenharia ({PALAVRA_CHAVE})",
                "empresa": "Inovação Digital Ltda",
                "link": "https://adzuna.com/exemplo-vaga-lider",
                "descricao": "Coordenar squads de desenvolvimento ágil, arquitetura de sistemas e integração de APIs."
            }
        ]

    resultados = []
    for v in vagas:
        resultados.append({
            "Título da Vaga": v["titulo"],
            "Empresa": v["empresa"],
            "Link": v["link"],
            "Descrição Resumida": v["descricao"],
            "Aderência (%)": "",
            "Justificativa": ""
        })

    df_final = pd.DataFrame(resultados)
    print(f"🟢 [Adzuna] Coleta finalizada em memória: {len(df_final)} registros.")
    return df_final

## 📊 Consolidação de Vagas (LinkedIn + Adzuna)
Nesta seção, unificamos de forma consolidada os resultados gerados pelas plataformas LinkedIn e Adzuna.

In [ ]:
import os
import asyncio
import pandas as pd

def executar_pipeline_e_consolidar():
    # 1. Executa a busca do LinkedIn diretamente em memória
    print("🚀 Executando crawler do LinkedIn...")
    try:
        df_lk = asyncio.run(pipeline_principal_linkedin())
        df_lk['Fonte'] = 'LinkedIn'
    except Exception as e:
        print(f"⚠️ Falha ao obter dados do LinkedIn: {e}")
        df_lk = pd.DataFrame()

    # 2. Executa a busca do Adzuna diretamente em memória
    print("🚀 Executando API do Adzuna...")
    try:
        df_az = pipeline_adzuna()
        df_az['Fonte'] = 'Adzuna'
    except Exception as e:
        print(f"⚠️ Falha ao obter dados do Adzuna: {e}")
        df_az = pd.DataFrame()

    dfs = []
    if not df_lk.empty:
        dfs.append(df_lk)
    if not df_az.empty:
        dfs.append(df_az)

    if not dfs:
        print("❌ Nenhuma vaga foi capturada de nenhuma das fontes!")
        return None

    # Unifica os DataFrames em memória
    df_consolidado = pd.concat(dfs, ignore_index=True)

    # Remove possíveis duplicatas mantendo a primeira ocorrência
    total_antes = len(df_consolidado)
    df_consolidado = df_consolidado.drop_duplicates(subset=['Título da Vaga', 'Empresa'], keep='first')
    total_depois = len(df_consolidado)

    print(f"🔄 Consolidação realizada: {total_antes} vagas unificadas -> {total_depois} vagas únicas (removidas {total_antes - total_depois} duplicatas).")

    # Reordena colunas de maneira consistente
    colunas = ['Fonte', 'Título da Vaga', 'Empresa', 'Link', 'Descrição Resumida', 'Aderência (%)', 'Justificativa']
    colunas_finais = [c for c in colunas if c in df_consolidado.columns]
    df_consolidado = df_consolidado[colunas_finais]

    # Define caminhos e cria a pasta final apenas para o arquivo consolidado
    pasta_vagas = "/content/drive/MyDrive/Colab Notebooks/Vagas"
    os.makedirs(pasta_vagas, exist_ok=True)

    caminho_cons_excel = os.path.join(pasta_vagas, "vagas_consolidadas.xlsx")
    caminho_cons_csv = os.path.join(pasta_vagas, "vagas_consolidadas.csv")

    # Salva exclusivamente os arquivos consolidados
    df_consolidado.to_excel(caminho_cons_excel, index=False)
    df_consolidado.to_csv(caminho_cons_csv, index=False, encoding="utf-8-sig")

    print(f"\n💾 Arquivo consolidado de vagas gerado EXCLUSIVAMENTE em:")
    print(f"   - Excel: '{caminho_cons_excel}'")
    print(f"   - CSV: '{caminho_cons_csv}'")
    return df_consolidado

# Executa todo o fluxo de raspagem conjunta e consolidação direta
df_geral_atualizado = executar_pipeline_e_consolidar()
if df_geral_atualizado is not None:
    display(df_geral_atualizado.head(10))

In [57]:
import os

readme_content = """# 🤖 Robô de Coleta e Consolidação de Vagas (SRE / Tecnologia)

Este é um pipeline automatizado desenvolvido em Python para buscar, filtrar, unificar e consolidar vagas de emprego de múltiplas fontes (**LinkedIn** e **Adzuna**) com foco no **Brasil** e em oportunidades **Remotas**.

Ideal para profissionais e recrutadores que desejam monitorar o mercado de trabalho diariamente de forma automatizada e centralizada.

## 🚀 Funcionalidades

- **Varredura no LinkedIn**: Utiliza o crawler automatizado com `Playwright` para acessar as listagens públicas do LinkedIn de forma segura, simulando navegação humana (scroll), aplicando filtros geográficos estritos para o Brasil e identificando vagas com a flag de trabalho remoto (`f_WT=2`).
- **Consumo da API Adzuna**: Integração oficial com a API do Adzuna para obter vagas qualificadas em tempo real utilizando chaves de acesso.
- **Higiene e Consolidação de Dados**: Mescla os resultados obtidos de ambas as plataformas diretamente em memória, elimina registros duplicados de forma inteligente (mesmo título de vaga na mesma empresa) e gera de forma exclusiva uma base limpa.
- **Exportação Direta**: Gera arquivos prontos para consumo nos formatos Excel (`.xlsx`) e CSV (`.csv`).

## 📁 Estrutura de Arquivos Gerados

O robô mantém seu espaço limpo. Ele não cria arquivos temporários de buscas individuais. Apenas a saída final consolidada é gravada no seu diretório de preferência:

- `vagas_consolidadas.xlsx`: Planilha Excel ideal para análises manuais e filtros visuais.
- `vagas_consolidadas.csv`: Base formatada em CSV (UTF-8 com BOM) ideal para integrações com bancos de dados, BI (PowerBI/Looker Studio) ou pipelines de Inteligência Artificial.

## 🛠️ Tecnologias Utilizadas

- **Python 3.10+**
- **Playwright** (Navegador automatizado em modo headless para scraping)
- **BeautifulSoup4** & **Requests** (Análise sintática do HTML)
- **Pandas** & **OpenPyXL** (Manipulação de dados e exportação de relatórios)
- **Nest-Asyncio** (Suporte à execução assíncrona concorrente em ambientes baseados em notebooks)

## ⚙️ Configurações e Instalação

### Pré-requisitos

Antes de rodar, instale as dependências essenciais do sistema e os pacotes Python:

```bash
pip install playwright nest_asyncio pandas openpyxl requests beautifulsoup4
playwright install chromium
playwright install-deps
```

### Configuração de Parâmetros

No topo do script ou notebook, configure as variáveis de controle de busca:

```python
PALAVRA_CHAVE = "SRE remoto"
LOCALIDADE = "Brasil"
LIMITE_VAGAS = None # Defina um número ou None para trazer todas as vagas possíveis
```

## 📖 Como Executar

1. Abra o arquivo do Jupyter Notebook ou execute o script principal contendo a chamada:
   ```python
   df_vagas = executar_pipeline_e_consolidar()
   ```
2. Acesse a pasta configurada (por padrão no Google Drive em `Colab Notebooks/Vagas`) para visualizar sua planilha consolidada de vagas atualizada.

---
Desenvolvido para simplificar a busca por novas oportunidades de carreira. Sinta-se à vontade para contribuir, abrir Issues ou enviar pull requests!
"""

pasta_vagas = "/content/drive/MyDrive/Colab Notebooks/Vagas"
os.makedirs(pasta_vagas, exist_ok=True)
caminho_readme = os.path.join(pasta_vagas, "README.md")

with open(caminho_readme, "w", encoding="utf-8") as f:
    f.write(readme_content)

print(f"✅ README.md gerado com sucesso em: {caminho_readme}")


✅ README.md gerado com sucesso em: /content/drive/MyDrive/Colab Notebooks/Vagas/README.md
